In [3]:
# import torch
from ultralytics import YOLO

In [5]:
# Load the MBARI 315k model
model = YOLO("mbari_315k_yolov8.pt")

In [23]:
model.info(detailed=True)

layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d     False        2160       [80, 3, 3, 3]   -0.0826      8.27        float32
    0                       model.0.conv.bias              Conv2d     False          80                [80]      1.01       1.6        float32
    1                             model.0.act                SiLU     False           0                  []         -         -              -
    2                     model.1.conv.weight              Conv2d     False      115200     [160, 80, 3, 3]  -0.00149    0.0389        float32
    2                       model.1.conv.bias              Conv2d     False         160               [160]      1.72      1.62        float32
    3                 model.2.cv1.conv.weight              Conv2d     False       25600    [160, 160, 1, 1]  -0.00801    0.0557        float32
    3         

(112, 68604105, 0, 260.06712319999997)

In [24]:
model.names

{0: 'Abraliopsis',
 1: 'Abyssocucumis abyssorum',
 2: 'Acanthamunnopsis milleri',
 3: 'Acanthascinae',
 4: 'Acanthephyra',
 5: 'Acanthogorgia',
 6: 'Acanthoptilum',
 7: 'Acesta',
 8: 'Actinernus',
 9: 'Actiniaria',
 10: 'Actinopterygii',
 11: 'Aegina',
 12: 'Aeginura',
 13: 'Aequorea',
 14: 'Aglantha digitale',
 15: 'Akoya platinum',
 16: 'Albatrossia pectoralis',
 17: 'Alepisaurus ferox',
 18: 'Amblyraja hyperborea',
 19: 'Amphipoda',
 20: 'Anarrhichthys ocellatus',
 21: 'Anchinothria fissurata',
 22: 'Anguilliformes',
 23: 'Anoplogaster cornuta',
 24: 'Anoplopoma fimbria',
 25: 'Anotopterus nikparini',
 26: 'Anthoptilum grandiflorum',
 27: 'Antimora microlepis',
 28: 'Antipatharia',
 29: 'Apolemia',
 30: 'Aporocidaris milleri',
 31: 'Apostichopus',
 32: 'Appendicularia',
 33: 'Apristurus',
 34: 'Asbestopluma',
 35: 'Asbestopluma monticola',
 36: 'Asbestopluma rickettsi',
 37: 'Asteroidea',
 38: 'Asteronyx',
 39: 'Atolla',
 40: 'Aulosphaera',
 41: 'Aurelia aurita',
 42: 'Balticina cal

In [39]:
ckpt = getattr(model, "ckpt", None)
print("Has ckpt:", ckpt is not None)
if isinstance(ckpt, dict):
    for k in ["version", "date", "epoch", "license", "docs"]:
        print(k, ":", ckpt.get(k))
    print("All top-level keys:", sorted(ckpt.keys()))

Has ckpt: True
version : 8.0.149
date : 2023-08-10T20:52:57.336152
epoch : -1
license : None
docs : None
All top-level keys: ['best_fitness', 'date', 'ema', 'epoch', 'model', 'optimizer', 'train_args', 'updates', 'version']


In [42]:
def inspect_yolo_model(p):
    y = YOLO(str(p))
    print("task:", getattr(y, "task", None))
    print("model_name:", getattr(y, "model_name", None))
    print("checkpoint_path:", getattr(y, "ckpt_path", None))

    names = getattr(y.model, "names", None)
    if isinstance(names, dict):
        print("num_classes:", len(names))
        print("class_names_preview:", dict(list(names.items())[:10]))
    else:
        print("num_classes:", None)
        print("class_names_preview:", None)

    try:
        total_params = sum(t.numel() for t in y.model.parameters())
        trainable_params = sum(t.numel() for t in y.model.parameters() if t.requires_grad)
        print("total_params:", total_params)
        print("trainable_params:", trainable_params)
    except Exception as e:
        print("param_count_error:", e)

    try:
        stride = getattr(y.model, "stride", None)
        if stride is not None:
            if hasattr(stride, "tolist"):
                stride = stride.tolist()
            print("stride:", stride)
    except Exception as e:
        print("stride_error:", e)
    print()

    print("top_level_keys:", sorted(list(ckpt.keys())))

    # Common metadata fields seen in Ultralytics checkpoints
    wanted = [
        "version", "date", "epoch", "best_fitness", "license", "docs",
        "git", "ema", "updates"
    ]
    print()
    print("common_fields:")
    for k in wanted:
        v = ckpt.get(k, None)
        if k in ["model", "ema"] and v is not None:
            print(k + ":", type(v).__name__)
        else:
            print(k + ":", v)

    # train_args is often where useful training provenance lives
    train_args = ckpt.get("train_args", None)
    print()
    print("train_args_present:", isinstance(train_args, dict))
    if isinstance(train_args, dict):
        keep = [
            "model", "data", "imgsz", "epochs", "batch", "optimizer",
            "lr0", "lrf", "patience", "device", "project", "name",
            "seed", "close_mosaic", "resume", "pretrained"
        ]
        slim = {k: train_args.get(k) for k in keep if k in train_args}
        # print("train_args_selected:", json.dumps(slim, indent=2, default=str))

    # Class metadata sometimes appears here too
    print()
    print("extra_dataset_fields:")
    for k in ["names", "nc"]:
        v = ckpt.get(k, None)
        if isinstance(v, dict):
            print(k + "_count:", len(v))
            print(k + "_preview:", dict(list(v.items())[:10]))
        else:
            print(k + ":", v)

# Example use:
inspect_yolo_model("mbari_315k_yolov8.pt")

task: detect
model_name: mbari_315k_yolov8.pt
checkpoint_path: mbari_315k_yolov8.pt
num_classes: 499
class_names_preview: {0: 'Abraliopsis', 1: 'Abyssocucumis abyssorum', 2: 'Acanthamunnopsis milleri', 3: 'Acanthascinae', 4: 'Acanthephyra', 5: 'Acanthogorgia', 6: 'Acanthoptilum', 7: 'Acesta', 8: 'Actinernus', 9: 'Actiniaria'}
total_params: 68633145
trainable_params: 0
stride: [8.0, 16.0, 32.0]

top_level_keys: ['best_fitness', 'date', 'ema', 'epoch', 'model', 'optimizer', 'train_args', 'updates', 'version']

common_fields:
version: 8.0.149
date: 2023-08-10T20:52:57.336152
epoch: -1
best_fitness: None
license: None
docs: None
git: None
ema: None
updates: None

train_args_present: True

extra_dataset_fields:
names: None
nc: None
